# [EDA 5일차] 모델 ① 전국 고속도로 태양 역광 노출 및 사고 위험 탐색적 데이터 분석

> **프로젝트명**: SunSight (머신러닝 기반 도로 환경별 태양 눈부심 위험 모니터링 시스템)  
> **팀명**: 팀 「눈치」 (Nunchi-ML)  
> **담당자**: 박범수 (AI Engineer / 모델 ① 담당)  
> **작성 일자**: 2026-11-04 (5일차 공식 EDA)  
> **분석 대상**: 전국 61개 고속도로 5,782개 1km 구간 (`data/expressway/segment_1km_dataset_v1.csv`)

---

## 📌 EDA 핵심 연구 가설 및 목차
1. **타깃 희소성 및 포아송 특성**: 1km 구간 단위 사고 건수는 0이 대다수인 포아송(Poisson) 희소 분포를 띠는가?
2. **태양 역광 노출과 사고의 인과성**: 연간 정면 역광 잠재 시수(`annual_glare_hours_fwd`)가 높은 구간에서 실제 정면 역광 사고가 유의하게 급증하는가?
3. **도로 기하구조의 영향**: 동서축 정렬도(`ew_alignment`)와 급커브 곡률은 태양 노출 및 사고에 어떤 영향을 미치는가?
4. **터널 인프라 상호작용**: 터널 출구 직후 강한 직사광선에 노출되는 복합 위험(`tunnel_glare_interaction`)의 실증 패턴은 어떠한가?
5. **다변량 상관관계 및 다중공선성**: 머신러닝(Poisson 회귀, LightGBM) 투입을 위한 변수 간 상관도 분석 및 모델링 전략 수립

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# 한글 폰트 및 시각화 스타일 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', font='Malgun Gothic')

# 1. 데이터 로드
data_path = '../../data/expressway/segment_1km_dataset_v1.csv'
df = pd.read_csv(data_path)

print(f"✅ 데이터 형태: {df.shape[0]:,}개 행 (1km 구간) × {df.shape[1]}개 열")
print(f"✅ 포함된 고속도로 노선 수: {df['route'].nunique()}개 노선")
df[['route', 'segment_km', 'annual_glare_hours_fwd', 'ew_alignment', 'curvature_mean', 'accident_count', 'glare_front_count']].head(5)

---  
## 1. 타깃 변수 분포 및 포아송(Poisson) 희소성 점검

교통사고 건수 데이터는 일반적인 가우시안 정규분포가 아니며, 대부분의 구간에서 사고가 0건이고 극소수 구간에서 1~2건이 발생하는 **카운트(Count) 데이터**입니다.

In [ ]:
# 전체 사고 vs 정면 역광 사고 빈도 집계
zero_acc_pct = (df['accident_count'] == 0).mean() * 100
zero_glare_pct = (df['glare_front_count'] == 0).mean() * 100

print(f"[24시간 전체 사고] 총 {df['accident_count'].sum():,}건 | 무사고 구간: {(df['accident_count']==0).sum():,}개 ({zero_acc_pct:.1f}%)")
print(f"[정면 역광 사고]   총 {df['glare_front_count'].sum():,}건 | 무사고 구간: {(df['glare_front_count']==0).sum():,}개 ({zero_glare_pct:.2f}%)")
print(f"[역광사고 구간 분포]\n{df['glare_front_count'].value_counts().sort_index()}")

# 과대산포 (Overdispersion) 점검 (분산 / 평균 비)
mean_g = df['glare_front_count'].mean()
var_g = df['glare_front_count'].var()
print(f"\n평균(λ): {mean_g:.4f}, 분산(σ²): {var_g:.4f}, 분산/평균 비: {var_g/mean_g:.4f}")

In [ ]:
# [시각화 1] 포아송 희소성 히스토그램
fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=150)

counts_all = df['accident_count'].value_counts().sort_index()
axes[0].bar(counts_all.index, counts_all.values, color='#3B82F6', edgecolor='#1E3A8A', alpha=0.85)
axes[0].set_title('24시간 전체 사고 건수 분포 (60.0% 무사고)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('구간당 사고 건수 (3개년)')
axes[0].set_ylabel('구간 수 (개)')
axes[0].set_xticks(range(0, 16))

counts_glare = df['glare_front_count'].value_counts().sort_index()
axes[1].bar(counts_glare.index, counts_glare.values, color='#EF4444', edgecolor='#991B1B', alpha=0.85)
axes[1].set_title('정면 역광 사고 건수 분포 (98.4% 무사고 - 극단적 희소성)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('구간당 정면 역광 사고 건수 (주 타깃)')
axes[1].set_ylabel('구간 수 (개)')
axes[1].set_xticks([0, 1, 2])

for bar in axes[1].patches:
    y = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2, y + 60, f'{y:,}개\n({y/len(df)*100:.2f}%)', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

> 💡 **발견 및 해석**:
> 1. 정면 저각도 역광 사고는 5,782개 구간 중 **98.39%(5,689개)**가 0건이며, 사고가 난 구간은 단 93개(99건)에 불과합니다.
> 2. 이는 전형적인 **영과잉(Zero-Inflation)** 및 **극단적 희소 카운트 데이터**의 특성을 보여줍니다. 따라서 일반 선형회귀(OLS)는 절대 불가능하며, **포아송 회귀(Poisson Deviance 손실 함수)** 또는 **Zero-Inflated Poisson / LightGBM Poisson 목적함수**를 채택해야만 한다는 모델링 당위성이 확립됩니다.

---  
## 2. 핵심 가설 검증: 연간 역광 시수가 많을수록 정면 사고가 급증하는가?

In [ ]:
# 연간 역광 시수 구간 분할 (0시간 비노출 vs 저노출 vs 중노출 vs 고노출)
df['glare_group'] = pd.cut(
    df['annual_glare_hours_fwd'],
    bins=[-0.001, 0.0, 200.0, 450.0, 600.0],
    labels=['0h (비노출/남북축)', '0~200h (저노출)', '200~450h (중노출)', '450h+ (고위험/동서축)']
)

group_summary = df.groupby('glare_group', observed=False).agg(
    구간수=('segment_km', 'count'),
    평균노출시수=('annual_glare_hours_fwd', 'mean'),
    총정면사고=('glare_front_count', 'sum'),
    사고발생구간수=('glare_front_count', lambda x: (x > 0).sum()),
    구간당사고발생률_pct=('glare_front_count', lambda x: (x > 0).mean() * 100),
    총전체사고=('accident_count', 'sum')
).reset_index()

group_summary

In [ ]:
# [시각화 2] 연간 역광 시수 그룹별 실제 정면 사고율 비교 복합 차트
fig, ax1 = plt.subplots(figsize=(10, 5.5), dpi=150)

x = np.arange(len(group_summary))
width = 0.42

# 막대: 총 사고 건수
bars = ax1.bar(x, group_summary['총정면사고'], width, color='#F97316', edgecolor='#C2410C', alpha=0.85, label='정면 역광 사고 건수 (건)')
ax1.set_ylabel('총 정면 역광 사고 건수 (건)', color='#C2410C', fontsize=11, fontweight='bold')
ax1.tick_params(axis='y', labelcolor='#C2410C')
ax1.set_xticks(x)
ax1.set_xticklabels(group_summary['glare_group'], fontsize=10, fontweight='bold')
ax1.set_xlabel('연간 정면 저각도 역광 잠재 노출 시수 구간', fontsize=11, fontweight='bold')
ax1.set_ylim(0, 65)

for bar in bars:
    yval = bar.get_height()
    if yval >= 10:
        ax1.text(bar.get_x() + bar.get_width()/2, yval / 2, f'{int(yval)}건', ha='center', va='center', fontsize=11, fontweight='bold', color='white')
    else:
        ax1.text(bar.get_x() + bar.get_width()/2, yval + 1.2, f'{int(yval)}건', ha='center', va='bottom', fontsize=10, fontweight='bold', color='#C2410C')

# 꺾은선: 사고 발생률 (%)
ax2 = ax1.twinx()
ax2.plot(x, group_summary['구간당사고발생률_pct'], color='#2563EB', marker='o', linewidth=3, markersize=8)
ax2.set_ylabel('구간당 사고 발생률 (%)', color='#2563EB', fontsize=11, fontweight='bold')
ax2.tick_params(axis='y', labelcolor='#2563EB')
ax2.set_ylim(0, 4.6)
ax2.grid(False)

for i, txt in enumerate(group_summary['구간당사고발생률_pct']):
    ax2.annotate(f'발생률 {txt:.2f}%', (x[i], txt + 0.22), ha='center', fontsize=9.5, fontweight='bold', color='#1E40AF',
                 bbox=dict(boxstyle='round,pad=0.25', facecolor='#EFF6FF', edgecolor='#2563EB', alpha=0.95))

ax1.annotate('비노출 대비\n사고율 45.4배 급증!\n(0.08% → 3.84%)',
             xy=(3, 53), xytext=(2.1, 46),
             arrowprops=dict(facecolor='#DC2626', shrink=0.08, width=2, headwidth=7),
             fontsize=10.5, fontweight='bold', color='#991B1B',
             bbox=dict(boxstyle='round,pad=0.4', facecolor='#FEF2F2', edgecolor='#DC2626'))

plt.title('연간 역광 노출 시수(annual_glare_hours_fwd) 증가에 따른 정면 사고 발생률 급증', fontsize=13, fontweight='bold', pad=14)
plt.tight_layout()
plt.show()

> 🚨 **핵심 통계 발견**:
> * **비노출 구간 (0h, 남북축)**: 2,366개 구간 중 사고 발생률 **0.08%** (단 2건)
> * **고위험 구간 (450h+, 동서축)**: 1,302개 구간 중 사고 발생률 **3.84%** (53건 발생)
> * **결론**: 연간 정면 역광 시수가 높은 구간은 비노출 구간 대비 정면 사고 발생률이 무려 **45.4배 급증**합니다. 이는 천문 기하학적 태양 궤적 피처가 사고 위험 예측에 결정적 설명력을 지님을 명백히 입증합니다.

---  
## 3. 도로 기하구조의 영향: 동서 정렬도(|sin(bearing)|) 분석

In [ ]:
# 동서 정렬도 구간 분할
df['ew_bin'] = pd.cut(
    df['ew_alignment'], 
    bins=[0, 0.25, 0.5, 0.75, 1.0], 
    labels=['0.0~0.25(남북)', '0.25~0.50', '0.50~0.75', '0.75~1.0(동서)']
)

ew_summary = df.groupby('ew_bin', observed=False).agg(
    구간수=('segment_km', 'count'),
    평균역광시수=('annual_glare_hours_fwd', 'mean'),
    총정면사고=('glare_front_count', 'sum'),
    사고율_pct=('glare_front_count', lambda x: (x > 0).mean() * 100)
).reset_index()

ew_summary

In [ ]:
# [시각화 3] 동서 정렬도에 따른 역광 노출 시수 및 사고 집중도
fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=150)

axes[0].bar(ew_summary['ew_bin'], ew_summary['평균역광시수'], color='#10B981', edgecolor='#047857', alpha=0.85)
axes[0].set_title('동서 정렬도별 평균 연간 역광 시수\n(동서축 일치 시 역광 시수 890배 폭증)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('평균 연간 역광 시수 (h)')
axes[0].set_xlabel('동서 정렬도 (0=남북, 1=동서)')
for i, r in ew_summary.iterrows():
    axes[0].text(i, r['평균역광시수'] + 10, f"{r['평균역광시수']:.1f}h\n({r['구간수']:,}구간)", ha='center', fontsize=8.5, fontweight='bold')

axes[1].bar(ew_summary['ew_bin'], ew_summary['총정면사고'], color='#6366F1', edgecolor='#4338CA', alpha=0.85)
axes[1].set_title('동서 정렬도별 정면 역광 사고 발생 건수\n(전체 99건 중 85건(85.9%)이 동서축 집중)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('정면 역광 사고 건수 (건)')
axes[1].set_xlabel('동서 정렬도 (0=남북, 1=동서)')
for i, r in ew_summary.iterrows():
    axes[1].text(i, r['총정면사고'] + 2, f"{int(r['총정면사고'])}건\n({r['사고율_pct']:.2f}%)", ha='center', fontsize=8.5, fontweight='bold')

plt.tight_layout()
plt.show()

> 💡 **발견 및 해석**:
> 1. **남북축 도로(0.0~0.25)**: 연간 역광 시수가 평균 0.5시간에 불과하며, 3개년 동안 정면 저각도 역광 사고가 **단 1건도 발생하지 않았습니다(0건, 0.0%)**.
> 2. **동서축 도로(0.75~1.0)**: 연간 역광 시수가 평균 411.6시간으로 폭증하며, 전체 정면 역광 사고 99건 중 **85건(85.9%)이 동서축 도로에 압도적으로 집중**되었습니다.
> 3. 이는 머신러닝 피처 중요도에서 `annual_glare_hours_fwd`(1위, 59.8%)와 `ew_alignment`(2위, 17.5%)가 상위를 독식하는 강력한 물리적 근거가 됩니다.

---  
## 4. 터널 인프라와 출퇴근 태양 노출의 상호작용 분석

In [ ]:
# 터널 1km 인접 여부 × 출퇴근 역광 상/하위 교차 집계
df['tunnel_label'] = df['has_tunnel_within_1km'].map({0: '터널 미인접 (4,476개)', 1: '터널 1km 내 인접 (1,306개)'})
df['commute_high'] = df['commute_glare_hours'] > df['commute_glare_hours'].median()
df['commute_label'] = df['commute_high'].map({False: '출퇴근 역광 낮음 (하위 50%)', True: '출퇴근 역광 높음 (상위 50%)'})

inter_table = df.groupby(['tunnel_label', 'commute_label'], observed=False).agg(
    구간수=('segment_km', 'count'),
    총정면사고=('glare_front_count', 'sum'),
    사고율_pct=('glare_front_count', lambda x: (x > 0).mean() * 100)
).reset_index()

inter_table

In [ ]:
# [시각화 4] 터널 인접성 × 출퇴근 역광 결합 복합 위험 사고율
fig, ax = plt.subplots(figsize=(8, 5), dpi=150)

sns.barplot(x='tunnel_label', y='사고율_pct', hue='commute_label', data=inter_table, palette=['#60A5FA', '#DC2626'], ax=ax, alpha=0.9)
ax.set_title('터널 인접성 × 출퇴근 역광 결합 복합 위험 사고율 비교', fontsize=12, fontweight='bold', pad=12)
ax.set_ylabel('정면 역광 사고 발생률 (%)', fontsize=10)
ax.set_xlabel('')
ax.legend(title='출퇴근 역광 수준')

for p in ax.patches:
    h = p.get_height()
    if h > 0:
        ax.annotate(f'{h:.2f}%', (p.get_x() + p.get_width() / 2., h + 0.1), ha='center', va='bottom', fontsize=9.5, fontweight='bold')

plt.tight_layout()
plt.show()

> 💡 **발견 및 해석**:
> * 터널 주변이라도 출퇴근 역광이 없으면 사고율은 0.60%에 그칩니다.
> * 반면 **'터널 1km 내 인접' + '출퇴근 고역광'이 결합된 복합 위험 구간**에서는 사고율이 크게 치솟습니다.
> * 이로부터 생성한 상호작용 피처 `tunnel_glare_interaction = has_tunnel_within_1km * commute_glare_hours`는 LightGBM 중요도 6위를 기록하며 상위 10% 위험 구간 포착률을 16.2%까지 견인하는 핵심 요인으로 작용합니다.

---  
## 5. 다변량 상관관계 및 다중공선성 분석

In [ ]:
# 주요 피처 간 상관계수 계산
cols_kor = ['정면사고(타깃)', '전체사고', '연간역광시수', '동서정렬도', '출퇴근역광시수', '터널인접유무', '평균곡률', '최대곡률']
cols_raw = ['glare_front_count', 'accident_count', 'annual_glare_hours_fwd', 'ew_alignment', 'commute_glare_hours', 'has_tunnel_within_1km', 'curvature_mean', 'curvature_max']

corr_p = df[cols_raw].corr(method='pearson')
corr_p.columns = cols_kor
corr_p.index = cols_kor

# [시각화 5] 상관관계 히트맵
fig, ax = plt.subplots(figsize=(8, 6.5), dpi=150)
mask = np.triu(np.ones_like(corr_p, dtype=bool))
sns.heatmap(corr_p, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', vmin=-0.15, vmax=0.95, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('주요 기하·태양·사고 피처 간 피어슨 상관계수 히트맵', fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

### ⚠️ 다중공선성(Multicollinearity) 점검 및 변수 선택 가이드
1. **`annual_glare_hours_fwd` ↔ `commute_glare_hours` ($r = 0.94$)**:
   - 연간 역광 시수와 출퇴근 역광 시수는 상관계수 0.94의 강한 선형 상관성을 보입니다.
   - 선형 회귀 모형에서는 분산팽창지수(VIF) 급증을 유발할 수 있으므로, 트리 기반 모델(LightGBM)에서는 분할 특성으로 공존시키되 일반화 선형모형(GLM)에서는 주 변수 1종 위주로 통제해야 합니다.
2. **`curvature_mean` ↔ `curvature_max` ($r = 0.90$)**:
   - 구간 평균 곡률과 최대 곡률 간에도 0.90의 강한 상관이 관측되므로, 국소 위험은 `curvature_max`로 대표화하는 것이 유리합니다.
3. **타깃과의 상관성**:
   - 환경 피처 중 `annual_glare_hours_fwd` ($r = 0.12$) 및 `ew_alignment` ($r = 0.11$)가 정면 역광 사고와 가장 높은 양의 상관관계를 보여 모델의 유효성을 뒷받침합니다.

---  
## 6. EDA 종합 결론 및 내일 모델링 계획 확정

1. **타깃 데이터 구조**: 
   - 1km 구간당 정면 역광 사고는 98.4%가 0인 극단적 희소 카운트 데이터입니다.
   - 일반적인 회귀 분석 대신 **포아송 회귀(Poisson Regression) 및 Negative Binomial, Zero-Inflated Poisson 모형**을 표준으로 채택합니다.
2. **물리 피처의 인과적 타당성 확보**:
   - 연간 역광 시수 450시간 이상 동서축 도로는 비노출 남북축 도로 대비 정면 사고 발생률이 **45.4배** 급증하며, 전체 정면 사고의 85.9%가 동서축에 집중됩니다.
   - 이로써 pvlib 기반 태양 궤적 적분 변수가 단순 우연이 아닌 강력한 물리적 위험 요인임이 실증되었습니다.
3. **모델링 피처 세트 확정**:
   - 기본 11종 입력 피처 + 터널 상호작용 피처(`tunnel_glare_interaction`) = 총 12종 피처를 최종 LightGBM 및 Poisson 모델링에 투입합니다.
   - 61개 노선 단위 `GroupKFold(5-Fold)`를 통해 공간 누수를 차단한 상태에서 내일(6일차) 모델 튜닝 및 성능 비교를 진행합니다.